# Notebook 2 — Data Cleaning (SQL + Python hybrid)

This notebook will tuern the three raw CSVs produced by `01_scraping.ipynb` into
**analysis-ready datasets** saved to `data/clean/`.

## Choosing the right tools:

I deliberately chose to use **SQL for set-based work** and **Python for numerical/statistical work**.
To do this I analysed each step I needed to make and chose the right tool:

| Step | Tool | Reason |
|------|------|--------|
| League-season aggregates (sum, avg, group-by)   | SQL    | `GROUP BY` is the natural idiom |
| Gini coefficient per league-season              | Python | numpy formula; painful in plain SQL |
| Ranking players within each Italy squad by value | SQL    | `ROW_NUMBER() OVER (PARTITION BY ...)` window fn |
| Joining Italy results to squad aggregates        | SQL    | `JOIN` is a one-liner |

The data is small (~4,000 rows total), so the choices are not about **performance**, the point is
using the right tool for each transformation. SQL runs inside an in-memory SQLite
database (`sqlite3` is in the Python stdlib, so not to add extra dependencies).

## Outputs

- `data/clean/league_season.csv` — one row per country-season: foreign %, total market value, Gini
- `data/clean/italy_squad_clean.csv` — one row per player-tournament: value rank within squad
- `data/clean/italy_final.csv` — one row per tournament: Italy's result + squad aggregates


## 0. Setup — load raw CSVs into in-memory SQLite

In [17]:
import sys, sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

sys.path.insert(0, str(Path('../src').resolve()))
from helpers import gini  # numpy-based Gini coefficient

RAW_DIR   = Path('../data/raw')
CLEAN_DIR = Path('../data/clean')
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Load raw tables
league_club_data        = pd.read_csv(RAW_DIR / 'league_club_data.csv')
italy_squad_selections  = pd.read_csv(RAW_DIR / 'italy_squad_selections.csv')
italy_results           = pd.read_csv(RAW_DIR / 'italy_results.csv')

# Create in-memory SQLite DB and register each DataFrame as a table
conn = sqlite3.connect(':memory:')
league_club_data.to_sql('league_club_data', conn, index=False, if_exists='replace')
italy_squad_selections.to_sql('italy_squad_selections', conn, index=False, if_exists='replace')
italy_results.to_sql('italy_results', conn, index=False, if_exists='replace')

# Sanity check: print number of rows in each table
for name, in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall():
    n = conn.execute(f'SELECT COUNT(*) FROM "{name}"').fetchone()[0]
    print(f'  {name:<28} {n:>6,} rows')

  league_club_data              3,681 rows
  italy_squad_selections          413 rows
  italy_results                    13 rows


---
## 1. League-season aggregates — SQL `GROUP BY` 

From `league_club_data` (one row per club-season), I compute the per-league-season totals.
This is the textbook case for SQL: simple `GROUP BY` with aggregates. The output
will have 10 countries × ~20 seasons ≈ 200 rows (one per league-season).

In [18]:
query = '''
SELECT
    country,
    season_start,
    season,
    COUNT(*)                                              AS num_clubs,
    SUM(squad_size)                                       AS total_players,
    SUM(foreign_players)                                  AS total_foreign,
    ROUND(100.0 * SUM(foreign_players)
                / NULLIF(SUM(squad_size), 0), 2)          AS pct_foreign,
    ROUND(AVG(avg_age), 2)                                AS avg_age,
    ROUND(SUM(squad_market_value_m), 2)                   AS total_market_value_m,
    ROUND(AVG(squad_market_value_m), 2)                   AS avg_club_value_m
FROM league_club_data
WHERE squad_market_value_m IS NOT NULL
GROUP BY country, season_start, season
ORDER BY country, season_start;
'''
league_season = pd.read_sql(query, conn)
print(f'league_season: {len(league_season)} rows')
league_season.head()

league_season: 200 rows


,country,season_start,season,num_clubs,total_players,total_foreign,pct_foreign,avg_age,total_market_value_m,avg_club_value_m
0,Belgium,2006,2006-2007,18,613,287,46.82,24.82,401.43,22.30
1,Belgium,2007,2007-2008,18,609,293,48.11,25.33,438.84,24.38
2,Belgium,2008,2008-2009,18,594,298,50.17,25.02,513.88,28.55
3,Belgium,2009,2009-2010,16,528,261,49.43,24.91,462.54,28.91
4,Belgium,2010,2010-2011,16,574,286,49.83,24.79,486.39,30.40


---
## 2. Gini coefficient per league-season — Python

The Gini formula requires iterating over sorted values with an index, this is easy in
numpy (`helpers.gini`), and awkward in SQL without recursive CTEs. We compute Gini
per league-season in pandas, then merge it back into `league_season` so the
final table has everything in one place.

In [19]:
ginis = (
    league_club_data
    .dropna(subset=['squad_market_value_m'])
    .groupby(['country', 'season_start'])['squad_market_value_m']
    .apply(gini)
    .reset_index(name='gini_coefficient')
)
ginis['gini_coefficient'] = ginis['gini_coefficient'].round(4)

league_season = league_season.merge(ginis, on=['country', 'season_start'], how='left')
print(f'league_season now has {len(league_season.columns)} columns, whhich are: {list(league_season.columns)}')
league_season.head()

league_season now has 11 columns, whhich are: ['country', 'season_start', 'season', 'num_clubs', 'total_players', 'total_foreign', 'pct_foreign', 'avg_age', 'total_market_value_m', 'avg_club_value_m', 'gini_coefficient']


,country,season_start,season,num_clubs,total_players,total_foreign,pct_foreign,avg_age,total_market_value_m,avg_club_value_m,gini_coefficient
0,Belgium,2006,2006-2007,18,613,287,46.82,24.82,401.43,22.30,0.3221
1,Belgium,2007,2007-2008,18,609,293,48.11,25.33,438.84,24.38,0.3237
2,Belgium,2008,2008-2009,18,594,298,50.17,25.02,513.88,28.55,0.3830
3,Belgium,2009,2009-2010,16,528,261,49.43,24.91,462.54,28.91,0.3703
4,Belgium,2010,2010-2011,16,574,286,49.83,24.79,486.39,30.40,0.3567


In [20]:
# Save and re-register in SQLite so downstream SQL steps can reference it
out = CLEAN_DIR / 'league_season.csv'
league_season.to_csv(out, index=False)
league_season.to_sql('league_season', conn, index=False, if_exists='replace')
print(f'Saved {out} ({len(league_season)} rows)')

Saved ../data/clean/league_season.csv (200 rows)


---
## 3. Italy squad and value ranking — SQL window function

For each tournament, I will rank each player by market value so
concentration metrics can be achieved (top value, top-11 value) per squad. The window function
`ROW_NUMBER() OVER (PARTITION BY year ORDER BY market_value_m DESC)` is exactly
what SQL is good at.

Note that the raw file has a `club` column that is only populated for the two
manually-entered qualifier squads (2018/2022) — it's not needed for any
downstream analysis, so I don't select it.

In [21]:
query = '''
SELECT
    year,
    tournament,
    player,
    position,
    market_value_m,
    ROW_NUMBER() OVER (PARTITION BY year
                       ORDER BY market_value_m DESC NULLS LAST) AS value_rank,
    COUNT(*)     OVER (PARTITION BY year)                       AS squad_size
FROM italy_squad_selections
ORDER BY year, value_rank;
'''
italy_squad_clean = pd.read_sql(query, conn)
print(f'italy_squad_clean: {len(italy_squad_clean)} rows')

out = CLEAN_DIR / 'italy_squad_clean.csv'
italy_squad_clean.to_csv(out, index=False)
italy_squad_clean.to_sql('italy_squad_clean', conn, index=False, if_exists='replace')
print(f'Saved {out}')
italy_squad_clean.head()

italy_squad_clean: 413 rows
Saved ../data/clean/italy_squad_clean.csv


,year,tournament,player,position,market_value_m,value_rank,squad_size
0,2006,World Cup,Francesco Totti,NaN,35.0,1,44
1,2006,World Cup,Alessandro Nesta,NaN,33.0,2,44
2,2006,World Cup,Gianluca Zambrotta,NaN,28.0,3,44
3,2006,World Cup,Gennaro Gattuso,NaN,25.0,4,44
4,2006,World Cup,Antonio Cassano,NaN,25.0,5,44


---
## 4. Italy tournament-level view -SQL `JOIN` + aggregates

For each tournament year we want:

- The number of valued players in the squad
- Average, sum, max of the market value
- Average market value of the top-11 players (this stands in place for the starting 11 squad)
- Italy's actual result (via `JOIN` with `italy_results`)

This is the master table for modelling (OLS using `performance_score` as outcome).


In [22]:
query = '''
WITH squad_agg AS (
    SELECT
        year,
        tournament,
        COUNT(*)                                              AS players,
        ROUND(AVG(market_value_m), 2)                         AS avg_value_m,
        ROUND(SUM(market_value_m), 2)                         AS total_value_m,
        ROUND(MAX(market_value_m), 2)                         AS top_value_m,
        ROUND(AVG(CASE WHEN value_rank <= 11 THEN market_value_m END), 2)
                                                              AS avg_top11_value_m
    FROM italy_squad_clean
    WHERE market_value_m IS NOT NULL
    GROUP BY year, tournament
)
SELECT
    r.year,
    r.tournament,
    r.result,
    r.performance_score,
    r.qualified,
    r.host,
    s.players,
    s.avg_value_m,
    s.total_value_m,
    s.top_value_m,
    s.avg_top11_value_m
FROM italy_results r
LEFT JOIN squad_agg s
    -- Italy didn't qualify in 2018/2022, so the matching squad row has
    -- tournament = 'World Cup (Qualifier)'. Join on year alone (unique per year).
    ON r.year = s.year
WHERE r.year >= 2006    -- only years for which we have squad data
ORDER BY r.year;
'''
italy_final = pd.read_sql(query, conn)
print(f'italy_final: {len(italy_final)} rows')
italy_final

italy_final: 11 rows


,year,tournament,result,performance_score,qualified,host,players,avg_value_m,total_value_m,top_value_m,avg_top11_value_m
0,2006,World Cup,Winner,7,1,Germany,42.0,9.03,379.35,35.0,23.18
1,2008,Euro,Quarter-final,4,1,Austria/Switzerland,45.0,9.37,421.60,30.0,21.14
2,2010,World Cup,Group Stage,2,1,South Africa,47.0,10.23,480.65,36.0,22.50
3,2012,Euro,Runner-up,6,1,Poland/Ukraine,39.0,12.07,470.90,38.0,22.36
4,2014,World Cup,Group Stage,2,1,Brazil,47.0,10.80,507.80,34.0,21.14
5,2016,Euro,Quarter-final,4,1,France,47.0,10.33,485.30,30.0,20.32
6,2018,World Cup,Did Not Qualify,1,0,Russia,19.0,23.87,453.50,70.0,36.45
7,2021,Euro,Winner,7,1,Various,49.0,25.08,1229.00,75.0,52.73
8,2022,World Cup,Did Not Qualify,1,0,Qatar,22.0,30.23,665.00,60.0,47.27
9,2024,Euro,Round of 16,3,1,Germany,49.0,20.70,1014.50,70.0,46.36


In [23]:
out = CLEAN_DIR / 'italy_final.csv'
italy_final.to_csv(out, index=False)
print(f'Saved {out} ({len(italy_final)} rows)')

Saved ../data/clean/italy_final.csv (11 rows)


---
## Final check

All three clean files should exist and be populated.

In [24]:
files = ['league_season.csv', 'italy_squad_clean.csv', 'italy_final.csv']
all_ok = True
for fname in files:
    fpath = CLEAN_DIR / fname
    if fpath.exists():
        df = pd.read_csv(fpath)
        print(f'  ok      {fname:<30} {len(df):>4} rows, {len(df.columns):>2} cols')
    else:
        print(f'  MISSING {fname}')
        all_ok = False

conn.close()
print('\nAll clean')

  ok      league_season.csv               200 rows, 11 cols
  ok      italy_squad_clean.csv           413 rows,  7 cols
  ok      italy_final.csv                  11 rows, 11 cols

All clean
